# GulfLink  Attachment Pipeline 

1. Load `gulf_link_analysis_ready.csv`
2. Filter rows with attachments
3. Expand attachment URLs
4. Download files using browser-like headers that work with Regulations.gov
5. Extract text from PDFs with `pdfplumber`
6. Merge attachment text back into the analysis dataset
7. Export an LLM-ready CSV


## 1. Setup


In [1]:
from pathlib import Path
import hashlib
import re
import time

import numpy as np
import pandas as pd
import pdfplumber
import requests
from tqdm import tqdm

pd.set_option("display.max_colwidth", 160)


In [2]:
# Locate the analysis-ready file produced by notebook 01.
# Not committed to the repository (see data/README.md).
REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
CANDIDATE_INPUTS = [
    REPO_ROOT / "data" / "gulf_link_analysis_ready.csv",
    REPO_ROOT / "data" / "gulf_link_analysis_ready.csv.gz",
    Path.cwd() / "gulf_link_analysis_ready.csv",
]

INPUT = next((path for path in CANDIDATE_INPUTS if path.exists()), None)
if INPUT is None:
    raise FileNotFoundError(
        "Could not find gulf_link_analysis_ready.csv. Run notebook 01 first, "
        "or see data/README.md. Expected one of:\n  "
        + "\n  ".join(str(p) for p in CANDIDATE_INPUTS)
    )

OUTPUT_DIR = Path.cwd()
CACHE_DIR = OUTPUT_DIR / "attachment_cache_fixed"
CACHE_DIR.mkdir(exist_ok=True)

print(f"Input:     {INPUT}")
print(f"Output dir:{OUTPUT_DIR.resolve()}")
print(f"Cache dir: {CACHE_DIR.resolve()}")

Input:     data/gulf_link_analysis_ready.csv
Output dir:data
Cache dir: data/attachment_cache_fixed


## 2. Load analysis ready data from the EDA notebook


In [3]:
analysis_df = pd.read_csv(INPUT, low_memory=False)
print(f"Loaded {len(analysis_df):,} rows")
print(f"Rows with attachments: {int(analysis_df['has_attachment'].sum()):,}")
analysis_df.head(2)


Loaded 10,195 rows
Rows with attachments: 321


,Document ID,Docket ID,Title,posted_date,posted_month,posted_year,First Name,Last Name,Organization Name,organization_clean,...,marine_wildlife,fisheries,wetlands_coast,national_interest,request_deny_or_reject,request_more_analysis,request_protect_wildlife,request_address_ej,Attachment Files,Content Files
0,MARAD-2019-0093-0004,MARAD-2019-0093,Comment from NELDA LYCKA,2019-07-03 04:00:00+00:00,2019-07,2019,NELDA,LYCKA,NaN,NaN,...,False,False,True,False,True,False,False,False,NaN,NaN
1,MARAD-2019-0093-0005,MARAD-2019-0093,Comment from Annette Hardesty,2019-07-03 04:00:00+00:00,2019-07,2019,Annette,Hardesty,NaN,NaN,...,True,False,True,False,False,False,True,False,NaN,NaN


## 3. Keep only submissions with attachments


In [4]:
att_df = analysis_df.loc[analysis_df["has_attachment"]].copy()
print(f"Submissions with attachments: {len(att_df):,}")
att_df[["Document ID", "attachment_count", "comment_text"]].head(10)


Submissions with attachments: 321


,Document ID,attachment_count,comment_text
3,MARAD-2019-0093-0007,20,Attached please find additional references cited in the comments submitted by the Center for Biological Diversity.
4,MARAD-2019-0093-0008,20,Attached please find comments from the Center for Biological Diversity on scoping for the Texas GulfLink Deepwater Port License Application. The references ...
5,MARAD-2019-0093-0009,5,Attached please find additional references cited in the comments submitted by the Center for Biological Diversity.
6,MARAD-2019-0093-0010,20,Attached please find additional references cited in the comments submitted by the Center for Biological Diversity.
7,MARAD-2019-0093-0011,20,Attached please find additional references cited in the comments submitted by the Center for Biological Diversity.
8,MARAD-2019-0093-0012,20,Attached please find additional references cited in the comments submitted by the Center for Biological Diversity.
9,MARAD-2019-0093-0013,20,Attached please find additional references cited in the comments submitted by the Center for Biological Diversity.
15,MARAD-2019-0093-0023,1,"Attached please find a petition signed by the residents of Jones Creek, Texas in opposition to this project."
25,MARAD-2019-0093-0039,1,"Comments of Texas Parks and Wildlife Department for the Deepwater Port License Application for Texas GulfLink, LLC are attached."
208,MARAD-2019-0093-0273,1,Attached


## 4. Expand attachment URLs to one row per attachment


In [5]:
rows = []
for _, row in att_df.iterrows():
    raw_urls = str(row["Attachment Files"]).split(",")
    urls = [u.strip() for u in raw_urls if u.strip()]
    for url in urls:
        rows.append(
            {
                "Document ID": row["Document ID"],
                "posted_date": row.get("posted_date"),
                "state_clean": row.get("state_clean"),
                "organization_clean": row.get("organization_clean"),
                "comment_text": row.get("comment_text", ""),
                "attachment_url": url,
            }
        )

attachments_long = pd.DataFrame(rows)
print(f"Total attachment URLs: {len(attachments_long):,}")
attachments_long.head()


Total attachment URLs: 622


,Document ID,posted_date,state_clean,organization_clean,comment_text,attachment_url
0,MARAD-2019-0093-0007,2019-07-11 04:00:00+00:00,NaN,Center For Biological Diversity,Attached please find additional references cited in the comments submitted by the Center for Biological Diversity.,https://downloads.regulations.gov/MARAD-2019-0093-0007/attachment_15.pdf
1,MARAD-2019-0093-0007,2019-07-11 04:00:00+00:00,NaN,Center For Biological Diversity,Attached please find additional references cited in the comments submitted by the Center for Biological Diversity.,https://downloads.regulations.gov/MARAD-2019-0093-0007/attachment_13.pdf
2,MARAD-2019-0093-0007,2019-07-11 04:00:00+00:00,NaN,Center For Biological Diversity,Attached please find additional references cited in the comments submitted by the Center for Biological Diversity.,https://downloads.regulations.gov/MARAD-2019-0093-0007/attachment_11.pdf
3,MARAD-2019-0093-0007,2019-07-11 04:00:00+00:00,NaN,Center For Biological Diversity,Attached please find additional references cited in the comments submitted by the Center for Biological Diversity.,https://downloads.regulations.gov/MARAD-2019-0093-0007/attachment_2.pdf
4,MARAD-2019-0093-0007,2019-07-11 04:00:00+00:00,NaN,Center For Biological Diversity,Attached please find additional references cited in the comments submitted by the Center for Biological Diversity.,https://downloads.regulations.gov/MARAD-2019-0093-0007/attachment_20.pdf


In [6]:
def url_to_filename(url):
    match = re.search(r"/([^/]+)/(attachment_\d+\.[A-Za-z0-9]+)$", url)
    if match:
        return f"{match.group(1)}__{match.group(2)}"
    return hashlib.md5(url.encode("utf-8")).hexdigest()[:16] + ".bin"

attachments_long["local_filename"] = attachments_long["attachment_url"].map(url_to_filename)
attachments_long["local_path"] = attachments_long["local_filename"].map(lambda name: CACHE_DIR / name)
attachments_long["file_ext"] = attachments_long["local_filename"].map(lambda name: Path(name).suffix.lower())

print("File extension distribution:")
print(attachments_long["file_ext"].value_counts())


File extension distribution:
file_ext
.pdf     598
.jpg       9
.docx      9
.xlsx      6
Name: count, dtype: int64


## 5. Download attachments with browser-style headers

In [7]:
SESSION = requests.Session()
SESSION.headers.update(
    {
        "User-Agent": (
            "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
            "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/137.0 Safari/537.36"
        ),
        "Referer": "https://www.regulations.gov/",
        "Accept": "application/pdf,application/octet-stream,*/*",
        "Accept-Language": "en-US,en;q=0.9",
        "Connection": "keep-alive",
    }
)

def download_one(url, local_path, timeout=60, sleep_seconds=0.2):
    if local_path.exists() and local_path.stat().st_size > 0:
        return "cached"

    try:
        response = SESSION.get(url, timeout=timeout, allow_redirects=True)
        if response.status_code == 200:
            local_path.write_bytes(response.content)
            if sleep_seconds:
                time.sleep(sleep_seconds)
            return "ok"
        if sleep_seconds:
            time.sleep(sleep_seconds)
        return f"http_{response.status_code}"
    except Exception as exc:
        if sleep_seconds:
            time.sleep(sleep_seconds)
        return f"error:{type(exc).__name__}"

to_download = attachments_long.loc[attachments_long["file_ext"].eq(".pdf")].copy()
print(f"PDFs to download/check: {len(to_download):,}")

statuses = []
for _, row in tqdm(to_download.iterrows(), total=len(to_download), desc="Downloading PDFs"):
    statuses.append(download_one(row["attachment_url"], row["local_path"]))

to_download["download_status"] = statuses

print()
print("Download status counts:")
print(to_download["download_status"].value_counts())


PDFs to download/check: 598



Download status counts:
download_status
cached    598
Name: count, dtype: int64


In [8]:
download_failures = to_download.loc[
    ~to_download["download_status"].isin(["ok", "cached"]),
    ["Document ID", "attachment_url", "download_status"],
].copy()

print(f"Download failures: {len(download_failures):,}")
download_failures.head(20)


Download failures: 0


,Document ID,attachment_url,download_status


## 6. Extract text from downloaded PDFs


In [9]:
def extract_pdf_text(path, max_pages=80):
    try:
        text_parts = []
        with pdfplumber.open(path) as pdf:
            total_pages = len(pdf.pages)
            pages_to_read = min(total_pages, max_pages)
            for page in pdf.pages[:pages_to_read]:
                page_text = page.extract_text()
                if page_text:
                    text_parts.append(page_text)
        return "\n\n".join(text_parts), total_pages, None
    except Exception as exc:
        return "", 0, f"{type(exc).__name__}: {exc}"

extracted_texts = []
attachment_pages = []
extraction_errors = []

for _, row in tqdm(to_download.iterrows(), total=len(to_download), desc="Extracting PDF text"):
    if row["download_status"] in ("ok", "cached") and row["local_path"].exists():
        text, pages, error = extract_pdf_text(row["local_path"])
        extracted_texts.append(text)
        attachment_pages.append(pages)
        extraction_errors.append(error)
    else:
        extracted_texts.append("")
        attachment_pages.append(0)
        extraction_errors.append("not_downloaded")

to_download["attachment_text"] = extracted_texts
to_download["attachment_pages"] = attachment_pages
to_download["extraction_error"] = extraction_errors
to_download["attachment_text_length"] = to_download["attachment_text"].str.len()

downloaded_mask = to_download["download_status"].isin(["ok", "cached"])
extracted_mask = downloaded_mask & to_download["attachment_text_length"].gt(0)
ocr_candidate_mask = downloaded_mask & to_download["attachment_text_length"].eq(0)

print()
print("Extraction summary:")
print(f"  Downloaded PDFs: {int(downloaded_mask.sum()):,}")
print(f"  PDFs with text extracted: {int(extracted_mask.sum()):,}")
print(f"  Downloaded PDFs needing OCR / image fallback: {int(ocr_candidate_mask.sum()):,}")
print(f"  Median extracted text length: {to_download.loc[downloaded_mask, 'attachment_text_length'].median():.0f} chars")
print(f"  Total extracted characters: {int(to_download['attachment_text_length'].sum()):,}")


Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Cannot set non-stroke color: 2 components specified, but only 1 (grayscale), 3 (RGB), and 4 (CMYK) are supported
Cannot set non-stroke color: 2 components specified, but only 1 (grayscale), 3 (RGB), and 4 (CMYK) are supported
Cannot set non-stroke color: 2 components specified, but only 1 (grayscale), 3 (RGB), and 4 (CMYK) are supported
Cannot set non-stroke color: 2 components specified, but only 1 (grayscale), 3 (RGB), and 4 (CMYK) are supported
Cannot set non-stroke color: 2 components specified, but only 1 (grayscale), 3 (RGB), and 4 (CMYK) are supported
Cannot set non-stroke color: 2 components specified, but only 1 (grayscale), 3 (RGB), and 4


Extraction summary:
  Downloaded PDFs: 598
  PDFs with text extracted: 360
  Downloaded PDFs needing OCR / image fallback: 238
  Median extracted text length: 10678 chars
  Total extracted characters: 27,688,261


In [10]:
to_download.loc[
    downloaded_mask,
    ["Document ID", "attachment_url", "attachment_pages", "attachment_text_length", "extraction_error"],
].sort_values("attachment_text_length", ascending=False).head(20)


,Document ID,attachment_url,attachment_pages,attachment_text_length,extraction_error
190,MARAD-2019-0093-2783,https://downloads.regulations.gov/MARAD-2019-0093-2783/attachment_15.pdf,65,791071,None
372,MARAD-2019-0093-3061,https://downloads.regulations.gov/MARAD-2019-0093-3061/attachment_5.pdf,167,648415,None
60,MARAD-2019-0093-0010,https://downloads.regulations.gov/MARAD-2019-0093-0010/attachment_5.pdf,82,461157,None
191,MARAD-2019-0093-2783,https://downloads.regulations.gov/MARAD-2019-0093-2783/attachment_13.pdf,39,454422,None
197,MARAD-2019-0093-2783,https://downloads.regulations.gov/MARAD-2019-0093-2783/attachment_16.pdf,34,449307,None
294,MARAD-2019-0093-2788,https://downloads.regulations.gov/MARAD-2019-0093-2788/attachment_11.pdf,77,414873,None
295,MARAD-2019-0093-2788,https://downloads.regulations.gov/MARAD-2019-0093-2788/attachment_9.pdf,122,335194,None
161,MARAD-2019-0093-2274,https://downloads.regulations.gov/MARAD-2019-0093-2274/attachment_1.pdf,40,304127,None
344,MARAD-2019-0093-2929,https://downloads.regulations.gov/MARAD-2019-0093-2929/attachment_8.pdf,215,285497,None
345,MARAD-2019-0093-2929,https://downloads.regulations.gov/MARAD-2019-0093-2929/attachment_7.pdf,118,276039,None


## 7. Aggregate attachment text per document


In [11]:
per_doc = (
    to_download.groupby("Document ID")
    .agg(
        n_pdf_attachments=("attachment_url", "count"),
        n_pdf_downloaded=("download_status", lambda s: int(s.isin(["ok", "cached"]).sum())),
        n_pdf_with_text=("attachment_text_length", lambda s: int((s > 0).sum())),
        attachment_text_combined=("attachment_text", lambda parts: "\n\n===\n\n".join(p for p in parts if p)),
        total_pages=("attachment_pages", "sum"),
        total_attachment_chars=("attachment_text_length", "sum"),
    )
    .reset_index()
)

print(f"Documents with any PDF attachment rows: {len(per_doc):,}")
print(f"Documents with extracted attachment text: {int(per_doc['total_attachment_chars'].gt(0).sum()):,}")
per_doc.head()


Documents with any PDF attachment rows: 311
Documents with extracted attachment text: 76


,Document ID,n_pdf_attachments,n_pdf_downloaded,n_pdf_with_text,attachment_text_combined,total_pages,total_attachment_chars
0,MARAD-2019-0093-0007,20,20,20,NATIONAL WILDLIFE FEDERATION\nFOUR YEARS\nINTO THE GULF OIL DISASTER:\nSTILL WAITING\nFOR RESTORATION\n\nNATIONAL WILDLIFE FEDERATION\n2 FOUR YEARS INTO THE...,944,1319092
1,MARAD-2019-0093-0008,20,20,20,7/8/2019 America's Dangerous Pipelines\n(/)\n(/)\n(/)\nAMERICA'S DANGEROUS PIPELINES\nHOME (/) > CAMPAIGNS (../../CAMPAIGNS/) > AMERICAS DANGEROUS PIPELINES...,806,1137206
2,MARAD-2019-0093-0009,5,5,5,"EVOLUTION The man who linked SPACE Mountain guardian INNOVATION Do Nobel laureates OBITUARY Jerome Karle, crystal\necological isolation and asks astronomers...",460,393526
3,MARAD-2019-0093-0010,20,20,19,"Federal Lands Greenhouse Gas Emissions\nand Sequestration in the United States:\nEstimates for 2005–14\nBy Matthew D. Merrill, Benjamin M. Sleeter, Philip A...",579,1664599
4,MARAD-2019-0093-0011,20,20,20,"Climate Stabilization Targets: Emissions, Concentrations, and Impacts over Decades to Millennia\nCLIMATE\nSTABILIZATION\nTARGETS\nEmissions, Concentrations,...",1294,1902006


## 8. Merge attachment text back into the main dataset


In [12]:
merged = att_df.merge(per_doc, on="Document ID", how="left")

def combine_text(row):
    parts = []
    if isinstance(row["comment_text"], str) and row["comment_text"].strip():
        parts.append(row["comment_text"].strip())
    if isinstance(row.get("attachment_text_combined"), str) and row["attachment_text_combined"].strip():
        parts.append("--- ATTACHMENT TEXT ---")
        parts.append(row["attachment_text_combined"].strip())
    return "\n\n".join(parts)

merged["llm_input_text"] = merged.apply(combine_text, axis=1)
merged["llm_input_length"] = merged["llm_input_text"].str.len()

print(f"Merged attachment rows: {len(merged):,}")
print(f"Rows with merged attachment text: {int(merged['total_attachment_chars'].fillna(0).gt(0).sum()):,}")
print(f"Median llm_input_length: {merged['llm_input_length'].median():.0f}")
print(f"Max llm_input_length:    {merged['llm_input_length'].max():,}")


Merged attachment rows: 321
Rows with merged attachment text: 76
Median llm_input_length: 12
Max llm_input_length:    2,901,866


## 9. Export a full LLM-ready dataset


In [13]:
full = analysis_df.merge(
    merged[
        [
            "Document ID",
            "n_pdf_attachments",
            "n_pdf_downloaded",
            "n_pdf_with_text",
            "attachment_text_combined",
            "total_pages",
            "total_attachment_chars",
            "llm_input_text",
            "llm_input_length",
        ]
    ],
    on="Document ID",
    how="left",
)

full["llm_input_text"] = full["llm_input_text"].fillna(full["comment_text"])
full["llm_input_length"] = full["llm_input_text"].str.len()

llm_cols = [
    "Document ID",
    "posted_date",
    "state_clean",
    "organization_clean",
    "comment_text",
    "comment_length",
    "has_attachment",
    "attachment_count",
    "n_pdf_attachments",
    "n_pdf_downloaded",
    "n_pdf_with_text",
    "total_pages",
    "total_attachment_chars",
    "attachment_text_combined",
    "llm_input_text",
    "llm_input_length",
    "exact_form_letter",
    "template_family",
    "stance",
]
llm_cols = [col for col in llm_cols if col in full.columns]

CONTROL_CHARS = re.compile(r"[\x00-\x08\x0b\x0c\x0e-\x1f]")

def clean_text_for_export(x):
    if pd.isna(x):
        return x
    x = str(x)
    x = x.replace("\x00", "")              
    x = x.replace("\r\n", "\n").replace("\r", "\n")
    x = CONTROL_CHARS.sub(" ", x)         
    return x

TEXT_COLS = ["comment_text", "attachment_text_combined", "llm_input_text"]

for col in TEXT_COLS:
    if col in full.columns:
        full[col] = full[col].map(clean_text_for_export)

export_path = OUTPUT_DIR / "gulf_link_llm_ready_final.csv"
full[llm_cols].to_csv(export_path, index=False)
print(f"Saved LLM-ready dataset to: {export_path}")


Saved LLM-ready dataset to: data/gulf_link_llm_ready_final.csv


## 10. Summary


In [14]:
downloaded_mask = to_download["download_status"].isin(["ok", "cached"])
extracted_mask = downloaded_mask & to_download["attachment_text_length"].gt(0)
ocr_candidate_mask = downloaded_mask & to_download["attachment_text_length"].eq(0)

summary = {
    "Public submissions (from EDA)": len(analysis_df),
    "Submissions with attachments": int(analysis_df["has_attachment"].sum()),
    "Attachment URLs total": len(attachments_long),
    "PDFs attempted": len(to_download),
    "PDFs downloaded (ok+cached)": int(downloaded_mask.sum()),
    "PDF download failures": int((~downloaded_mask).sum()),
    "PDFs with text extracted": int(extracted_mask.sum()),
    "Downloaded PDFs needing OCR": int(ocr_candidate_mask.sum()),
    "Total attachment text chars": int(to_download["attachment_text_length"].sum()),
    "Documents with merged text": int(full["total_attachment_chars"].fillna(0).gt(0).sum()),
}

for key, value in summary.items():
    print(f"{key:35s}: {value:,}" if isinstance(value, int) else f"{key:35s}: {value}")


Public submissions (from EDA)      : 10,195
Submissions with attachments       : 321
Attachment URLs total              : 622
PDFs attempted                     : 598
PDFs downloaded (ok+cached)        : 598
PDF download failures              : 0
PDFs with text extracted           : 360
Downloaded PDFs needing OCR        : 238
Total attachment text chars        : 27,688,261
Documents with merged text         : 76


In [15]:
ocr_candidates = to_download[
    to_download['download_status'].isin(['ok', 'cached']) &
    (to_download['attachment_text_length'] == 0)
].copy()

ocr_candidates['filename'] = ocr_candidates['local_path'].astype(str).str.split('/').str[-1]

ocr_candidates = ocr_candidates.sort_values('Document ID')

ocr_candidates[['Document ID', 'filename', 'attachment_url', 'local_path']].to_csv(
    'ocr_candidates.csv', index=False
)

print(f"Total PDFs needing OCR: {len(ocr_candidates)}")
print(f"Saved list to: ocr_candidates.csv")
print()
print("First 10:")
ocr_candidates[['Document ID', 'filename']].head(10)

Total PDFs needing OCR: 238
Saved list to: ocr_candidates.csv

First 10:


,Document ID,filename
49,MARAD-2019-0093-0010,MARAD-2019-0093-0010__attachment_16.pdf
125,MARAD-2019-0093-0023,MARAD-2019-0093-0023__attachment_1.pdf
165,MARAD-2019-0093-2277,MARAD-2019-0093-2277__attachment_1.pdf
329,MARAD-2019-0093-2837,MARAD-2019-0093-2837__attachment_1.pdf
359,MARAD-2019-0093-2995,MARAD-2019-0093-2995__attachment_1.pdf
375,MARAD-2019-0093-3099,MARAD-2019-0093-3099__attachment_1.pdf
376,MARAD-2019-0093-3107,MARAD-2019-0093-3107__attachment_1.pdf
377,MARAD-2019-0093-3108,MARAD-2019-0093-3108__attachment_1.pdf
378,MARAD-2019-0093-3109,MARAD-2019-0093-3109__attachment_1.pdf
379,MARAD-2019-0093-3110,MARAD-2019-0093-3110__attachment_1.pdf


In [16]:
!pip install pymupdf
import fitz  

def extract_with_pymupdf(path):
    try:
        doc = fitz.open(path)
        text = "\n".join(page.get_text() for page in doc)
        doc.close()
        return text
    except Exception as e:
        return ""

In [17]:
import pandas as pd
from tqdm import tqdm
from pathlib import Path

ocr_candidates = to_download[
    to_download['download_status'].isin(['ok', 'cached']) &
    (to_download['attachment_text_length'] == 0)
].copy()

print(f"Total PDFs to attempt rescue: {len(ocr_candidates)}")

rescued_texts = []
for path in tqdm(ocr_candidates['local_path'], desc="PyMuPDF rescue"):
    text = extract_with_pymupdf(str(path))
    rescued_texts.append(text)

ocr_candidates['pymupdf_text'] = rescued_texts
ocr_candidates['pymupdf_text_length'] = ocr_candidates['pymupdf_text'].str.len()

rescued = ocr_candidates[ocr_candidates['pymupdf_text_length'] > 0]
still_empty = ocr_candidates[ocr_candidates['pymupdf_text_length'] == 0]

print(f"\n=== Rescue results ===")
print(f"Rescued (now have text):  {len(rescued)} PDFs")
print(f"Still empty (need OCR):   {len(still_empty)} PDFs")
print(f"Total characters rescued: {ocr_candidates['pymupdf_text_length'].sum():,}")

still_empty[['Document ID', 'attachment_url', 'local_path']].to_csv(
    'still_need_ocr.csv', index=False
)
print(f"\nSaved 'still_need_ocr.csv' with {len(still_empty)} entries")

if len(rescued) > 0:
    print(f"\n=== Sample rescued texts (first 3) ===")
    for _, row in rescued.head(3).iterrows():
        print(f"\n--- {row['Document ID']} ---")
        print(row['pymupdf_text'][:300] + "...")

Total PDFs to attempt rescue: 238


PyMuPDF rescue: 100%|████████████████████████| 238/238 [00:00<00:00, 529.58it/s]


=== Rescue results ===
Rescued (now have text):  15 PDFs
Still empty (need OCR):   223 PDFs
Total characters rescued: 136

Saved 'still_need_ocr.csv' with 223 entries

=== Sample rescued texts (first 3) ===

--- MARAD-2019-0093-0010 ---









































...

--- MARAD-2019-0093-0023 ---





















...

--- MARAD-2019-0093-2277 ---























































...


In [18]:
# Filter still_need_ocr.csv by priority:
# 1. Submissions from organizations (not individuals)
# 2. Files > 200KB (likely actual documents, not signature pages)
# 3. First attachment of a submission (usually the main letter)

import os
still_empty['file_size_kb'] = still_empty['local_path'].apply(
    lambda p: os.path.getsize(p) / 1024 if os.path.exists(p) else 0
)

# Join with analysis_df to see who submitted
high_priority = still_empty.merge(
    analysis_df[['Document ID', 'Organization Name', 'organization_clean']],
    on='Document ID',
    how='left'
)

# Priority criteria
high_priority['is_org'] = high_priority['Organization Name'].notna()
high_priority['is_large'] = high_priority['file_size_kb'] > 200
high_priority['is_first_attachment'] = high_priority['local_filename'].str.contains('attachment_1.pdf', na=False)

high_priority['priority_score'] = (
    high_priority['is_org'].astype(int) * 2 +
    high_priority['is_large'].astype(int) +
    high_priority['is_first_attachment'].astype(int)
)

# Top 30 by priority
top_ocr = high_priority.sort_values('priority_score', ascending=False).head(30)
top_ocr.to_csv('high_priority_ocr.csv', index=False)
print(f"Top 30 high-priority OCR candidates saved")
print(top_ocr[['Document ID', 'Organization Name', 'file_size_kb', 'priority_score']].head(15))

Top 30 high-priority OCR candidates saved
              Document ID Organization Name  file_size_kb  priority_score
86   MARAD-2019-0093-3196               NaN    293.897461               2
196  MARAD-2019-0093-3309               NaN    246.048828               2
8    MARAD-2019-0093-3114               NaN    210.865234               2
152  MARAD-2019-0093-3264               NaN     26.400391               1
141  MARAD-2019-0093-3253               NaN     40.339844               1
142  MARAD-2019-0093-3254               NaN    127.525391               1
143  MARAD-2019-0093-3255               NaN     19.917969               1
144  MARAD-2019-0093-3256               NaN     31.514648               1
145  MARAD-2019-0093-3257               NaN     41.765625               1
146  MARAD-2019-0093-3258               NaN     83.358398               1
147  MARAD-2019-0093-3259               NaN     27.166992               1
148  MARAD-2019-0093-3260               NaN     47.038086             

/var/folders/_d/vrx14z754yl0ly7257jj5g7m0000gq/T/ipykernel_43019/869164745.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  still_empty['file_size_kb'] = still_empty['local_path'].apply(
